# Operaciones de aprendizaje automático
## Maestría en Inteligencia Artificial Aplicada

### TC5061 | Proyecto MiBici: propuesta y base reproducible | Colaborativo

#### **Equipo:** 7
#### **Nombre:** Fernando Arreola Treviño
#### **Matrícula:** A01840331

## Objetivo
Notebook de **feature engineering** del proyecto MiBici.

La pregunta que me corresponde es la **P2 (clasificación)**: ¿Una estación estará en riesgo de quedarse sin bicicletas o sin anclajes disponibles en un periodo futuro?

La lógica que se reutilice se trasladará al módulo del proyecto; en este notebook se desarrolla y se documenta la evidencia que la justifica.

## Dataset
**Datos abiertos de MiBici**, el sistema de bicicleta pública del Área Metropolitana de Guadalajara. El portal publica:
- **Viajes por mes** (`datos_abiertos_AAAA_MM.csv`) desde diciembre de 2014. Cada registro es un viaje con identificador, datos del usuario, fecha y hora de inicio y fin, y estaciones de origen y destino.
- **Nomenclatura de estaciones** (`nomenclatura_AAAA_MM.csv`) con identificador, nombre, latitud, longitud y capacidad de cada estación.

- Fuente: [Datos abiertos de MiBici](https://mibici.net/es/datos-abiertos/)
- Rango analizado: **enero de 2020 a septiembre de 2026**, acordado por el equipo para homologar el EDA. Puede recortarse después con base en los hallazgos.

Se cuenta con lógica implementada para descargar los archivos en un rango especificado y en formato parquet, homologando la codificación y estandarización de fechas (`dataset.py`). Se usó este código para descargar los datos del 2020 al 2026 en la carpeta data/external.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import sys
import os
import glob
import pyarrow
from datetime import date
from pathlib import Path
import matplotlib.pyplot as plt
from pathlib import Path
import re

print("Python      :", sys.version.split()[0])
print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)
print("pyarrow     :", pyarrow.__version__)
print("matplotlib  :", matplotlib.__version__)

Python      : 3.11.5
pandas      : 3.0.5
numpy       : 1.26.4
pyarrow     : 25.0.1
matplotlib  : 3.11.2


## Configuración global


In [2]:
# Rutas del proyecto (el notebook se ejecuta desde notebooks/)
RUTA_PROYECTO = Path.cwd().parent
DATA_DIR = RUTA_PROYECTO / "data" / "processed"
ESTACIONES_DIR = RUTA_PROYECTO / "data" / "raw" / "estaciones"

# Horas sin servicio de MiBici
SIN_SERVICIO = [1, 2, 3, 4]

SEMILLA = 42

HORIZONTE_MIN =120
BIN_MIN = 30
H_BINS = HORIZONTE_MIN // BIN_MIN
ALFA = 0.5

## Carga de datos

Se retoman los datos guardados de la libreta anterior.

In [3]:
def leer_datos():
    parquet_file = os.path.join(DATA_DIR, "p2_datos_viajes.parquet")
    return pd.read_parquet(parquet_file)

In [4]:
df = leer_datos()

In [5]:
def leer_estaciones():
    csv_file = glob.glob(os.path.join(ESTACIONES_DIR, "*.csv"))[0]
    return pd.read_csv(csv_file)

In [6]:
estaciones_df = leer_estaciones()

In [7]:
estaciones_df = estaciones_df[estaciones_df["dpcapacity"] > 0]

## Feature Engineering

Retomando las conclusiones de la libreta 0.02-fat-exploracion-y-limpieza-p2:

El dataset no provee datos apropiados para realizar un análisis de cambios en inventario para una estación a lo largo del tiempo. Esto se debe a tres razones:
1. No está definido un punto de partida de inventario por estación.
2. Es de esperarse que cada estación tenga reabastecimientos, particularmente al analizar periodos largos de tiempo.
3. Incluso si se estimara un valor de inventario inicial para cada estación, con un valor de entre 0 y su capacidad máxima, el análisis hecho anteriormente revela un hecho importante: algunas estaciones son más populares como origen que como destino, y viceversa.

El problema, por lo tanto, se ajustará de la siguiente manera: Se clasificará una estación como en riesgo si en un futuro cercano, presenta un cambio proporcionalmente alto respecto a su inventario total. De esta forma se evita depender de datos de inventario longitudinales sólidos para cada estación, y solo se requiere analizar los cambios de inventario próximos. El riesgo a clasificar para cada estación e intervalo de tiempo será de quedarse sin bicicletas si el cambio es negativo, y si el cambio es positivo el riesgo será de quedarse sin anclajes disponibles. Si el cambio no es lo suficientemente grande para entrar en alguna de estas clasificaciones, se clasifica como estación no en riesgo.

Para hacer un análisis consistente, es necesario pasar de un dataframe de eventos (cada registro un evento), a un dataframe de cambios: cada registro es un intervalo de tiempo predefinido, e indica cuantas salidas y entradas ocurrieron en este intervalo. El intervalo se define a continuación:

In [8]:
def intervalo(ts, bin_min):
    b = ts.dt.floor(f"{bin_min}min")
    ultimo = b.dt.floor("D") + pd.Timedelta(hours=1) - pd.Timedelta(minutes=bin_min)
    return b.where(~b.dt.hour.isin(SIN_SERVICIO), ultimo)


Se define una función para obtener el dataframe de eventos, pero agrupado en intervalos de tiempo, o bins:

In [9]:
def obtener_dataframe_de_eventos_con_bins(viajes, bin_min):
    # Obtener salidas y entradas por intervalo de tiempo y estación
    salidas = viajes.groupby(["origen_id", intervalo(viajes.inicio_del_viaje, bin_min=bin_min)]).size().rename("salidas")
    entradas = viajes.groupby(["destino_id", intervalo(viajes.fin_del_viaje, bin_min=bin_min)]).size().rename("entradas")

    # Unificar salidas y entradas en dataframe de eventos
    salidas.index.names = entradas.index.names = ["estacion", "bin"]
    eventos = pd.concat([salidas, entradas], axis=1).fillna(0).astype("int32")
    
    return eventos

Ya teniendo los eventos, se convierte a dataframe de cambios. Esto implica agregar intervalos de tiempos que pudieran faltar entre el primer viaje y último viaje de cada estación. Se definen funciones para esto a continuación:

In [10]:
def generar_arreglo_de_bins(eventos, bin_min):
    
    # De la fecha minima a la fecha maxima del dataframe de eventos, con frecuencia de bin_min
    bins = pd.date_range(eventos.index.get_level_values("bin").min().floor("D"),
                     eventos.index.get_level_values("bin").max(), freq=f"{bin_min}min")
    
    # Remover los bins que corresponden a horas sin servicio
    bins = bins[~bins.hour.isin(SIN_SERVICIO)] 
    
    return bins

In [11]:
def obtener_dataframe_de_cambios(viajes, bin_min):
    # Obtener dataframe de eventos
    eventos = obtener_dataframe_de_eventos_con_bins(viajes, bin_min=bin_min)
    
    # Obtener arreglo de bins de tiempo
    bins = generar_arreglo_de_bins(eventos, bin_min=bin_min)
    
    # Expandir eventos para que cada estación tenga records de tiempo con frecuencia de bin_min
    rejilla = pd.MultiIndex.from_product([eventos.index.get_level_values("estacion").unique(), bins], names=["estacion", "bin"])
    cambios = eventos.reindex(rejilla, fill_value=0).reset_index()       # los ceros también son filas
    
    return cambios

In [12]:
cambios = obtener_dataframe_de_cambios(df, bin_min=BIN_MIN)

# Por definición, el número de salidas encontradas debe ser igual al número de salidas encontradas
# Y por lo mismo ambos deben ser igual al número de viajes
assert cambios.salidas.sum() == len(df) and cambios.entradas.sum() == len(df)

print("cambios:", cambios.shape)

cambios: (46539200, 4)


Teniendo este dataframe, se agrega la suma acumulada de cambios de inventario, agrupada por estación y día. Cada día habil se reinicia el conteo.

In [13]:
def agregar_suma_acumulada_a_cambios(cambios):
    # Se calcula el flujo neto de bicicletas (entradas - salidas) y el día de servicio de cada record
    cambios["neto"] = cambios.entradas - cambios.salidas
    cambios["dia"] = (cambios.bin - pd.Timedelta(hours=5)).dt.floor("D")   # día de servicio: 05:00 -> 00:59
    
    # Se eliminan las combinaciones de estaciones y días sin actividad
    act = cambios.groupby(["estacion", "dia"])[["salidas", "entradas"]].transform("sum").sum(axis=1)
    cambios = cambios[act > 0]
    
    # Ordenar cambios por estación y por tiempo discretizado (bins)
    cambios = cambios.sort_values(["estacion", "bin"]).reset_index(drop=True)
    
    # Calcular suma acumulada de movimientos netos (entradas - salidas) por estación
    cambios["cumsum_dia"] = cambios.groupby(["estacion", "dia"], sort=False)["neto"].cumsum()
    
    return cambios
    
    

In [14]:
cambios = agregar_suma_acumulada_a_cambios(cambios)

Se le integran datos de estación

In [15]:
def agregar_datos_de_estacion_a_cambios(cambios, estaciones):
# Agregar datos de estaciones a cambios
    cambios = cambios.merge(estaciones[["id","latitude","longitude","dpcapacity"]], left_on="estacion", right_on="id", how="left")
    cambios = cambios.drop("id", axis=1) # Quitar la columna de ID que viene del dataframe de estaciones (redundante)
    return cambios

In [16]:
cambios = agregar_datos_de_estacion_a_cambios(cambios, estaciones_df)

In [17]:
def obtener_mayores_cambios_en_proximos_h_bins(cambios, h):
    """Peor salida neta y peor entrada neta en los intervalos t+1..t+h (mismo día), como fracción de la capacidad.
    La intención es usar estas mediciones como base para estimar el riesgo de que que una estación se quede sin bicicletas
    o se llene en los próximos h*bin_freq minutos.
    """
    
    # Obtener suma acumulada de movimiento por estación, por día
    cumsums = cambios.groupby(["estacion", "dia"], sort=False)["cumsum_dia"]
    
    # Calcular cambio neto de un tiempo t, al tiempo k
    # Se obtiene una columna por cada valor de k entre t+1 y h (h columnas en total)
    cambios_futuros = pd.concat([cumsums.shift(-k) - cambios["cumsum_dia"] for k in range(1, h + 1)], axis=1)
    
    # De los h cambios netos calculados:
    # Se obtiene el más negativo y se guarda en la variable "vacia"
    # Se obtiene el más positivo y se guarda en la variable "llena"
    # Ambas se obtienen como fracción de la capacidad total
    vacia = (-cambios_futuros).max(axis=1, skipna=False).clip(lower=0) / cambios["dpcapacity"]
    llena = cambios_futuros.max(axis=1, skipna=False).clip(lower=0) / cambios["dpcapacity"]
    return  vacia, llena       # NaN cuando la ventana futura se sale del día de servicio
 
 
cambios["vacia"], cambios["llena"] = obtener_mayores_cambios_en_proximos_h_bins(cambios, H_BINS)

Para cada valor tanto de "vacía" como de "llena", si es mayor a alfa, se considera como clasificación positiva para su riesgo correspondiente (estación vacía, o estación llena). Alfa es un umbral que responde, ¿a partir de qué punto un cambio futuro de inventario de bicicletas se considera como un riesgo?

Se asume alfa == 0.5. Si un cambio representa más de la mitad de la capacidad de una estación, se considera riesgo.

Se evalúa cuantas estaciones tienen al menos una clase positiva:

In [18]:
def evaluar_riesgos_por_estacion(cambios_con_riesgos):
      d = cambios_con_riesgos.dropna(subset=["vacia", "llena"])
      riesgos = (d.vacia >= ALFA) | (d.llena >= ALFA)
      por_estacion = riesgos.groupby(d.estacion).sum().sort_values(ascending=False)
      print("% de estaciones con al menos 1 positivo:", round((por_estacion > 0).mean(), 3))
      print("% de positivos en el 10% de estaciones con más positivos:",
      round(por_estacion.head(int(len(por_estacion) * 0.1)).sum() / por_estacion.sum(), 3))

evaluar_riesgos_por_estacion(cambios)

% de estaciones con al menos 1 positivo: 0.794
% de positivos en el 10% de estaciones con más positivos: 0.56


Con esto, podemos crear finalmente la columna a predecir. Esta se denomina "clase".
- clase == 0: sin riesgo.
- clase == 1: riesgo de no tener bicicletas en el la ventana futura evaluada.
- clase == 2: riesgo de no tener anclajes libres en la ventana futura evaluada.

In [19]:
def agregar_clase_de_riesgo(cambios, alfa):
    """0 = sin_riesgo, 1 = sin_bicis, 2 = sin_anclajes. Si una ventana tiene ambos eventos, gana el de mayor estrés relativo."""
    datos = cambios.dropna(subset=["vacia", "llena"]).copy()          # fuera las filas cuya ventana futura se sale del día
    vacia_positiva, llena_positiva = datos.vacia >= alfa, datos.llena >= alfa
    datos["clase"] = 0
    datos.loc[vacia_positiva, "clase"] = 1
    datos.loc[llena_positiva, "clase"] = 2
    ambas = vacia_positiva & llena_positiva
    datos.loc[ambas, "clase"] = np.where(datos.loc[ambas, "vacia"] >= datos.loc[ambas, "llena"], 1, 2)
    print("ventanas con ambos eventos:", int(ambas.sum()), "de", int((vacia_positiva | llena_positiva).sum()), "positivas")
    print("distribución de clases:", datos.clase.value_counts(normalize=True).sort_index().round(4).to_dict())
    return datos

In [20]:
cambios = agregar_clase_de_riesgo(cambios, ALFA)

ventanas con ambos eventos: 7 de 666261 positivas
distribución de clases: {0: 0.9769, 1: 0.0114, 2: 0.0117}


Con este dataframe resultante, se generan características para el modelo.

Las siguientes características miden el cambio de capacidad hacia atrás en el tiempo:
- Se obtiene la diferencia en cambio de inventario neto, 1 bin de tiempo atrás, 2 bins de tiempo atrás, 4 bins de tiempo atrás, y 8 bins de tiempo atrás.
- Se obtiene la diferencia en salidas de bicicletas, 2 bins de tiempo atrás y 4 bins de tiempo atrás. Se hace lo mismo para las entradas de bicicletas.

Además, se divide la suma acumulada de cambios diarios entre la capacidad de cada estación, como paso de normalización.

La hora se introduce como feature: es posible que ciertas estaciones estén en mayor o menor riesgo de llenarse o vaciarse en ciertas horas del día. Para no perder la cualidad cíclica de las horas, se guarda su seno y coseno como características.

Finalmente, se agrega una variable binaria indicando si un record pertenece a un fin de semana.

In [21]:
def ventana(cambios, col, k):
    g = cambios.groupby(["estacion", "dia"], sort=False)[col]
    c = g.cumsum()
    return c - c.groupby([cambios.estacion, cambios.dia], sort=False).shift(k, fill_value=0)

In [22]:
def agregar_caracteristicas_retrospectivas(cambios):
    for k in (1, 2, 4, 8):
        cambios[f"neto_{k}_relativo"] = ventana(cambios, "neto", k) / cambios.dpcapacity
    for k in (2, 4):
        cambios[f"salidas_{k}"] = ventana(cambios, "salidas", k)
        cambios[f"entradas_{k}"] = ventana(cambios, "entradas", k)
    return cambios     

In [23]:
def agregar_caracteristicas_de_hora(cambios, bin_min):
    cambios["hora_idx"] = (cambios.bin.dt.hour * 60 + cambios.bin.dt.minute) // bin_min
    bins_por_dia = 60 * 24 / bin_min
    cambios["hora_sin"] = np.sin(2 * np.pi * cambios.hora_idx / bins_por_dia)
    cambios["hora_cos"] = np.cos(2 * np.pi * cambios.hora_idx / bins_por_dia)
    return cambios

In [24]:
cambios = agregar_caracteristicas_retrospectivas(cambios)

cambios["cumsum_dia_relativo"] = cambios.cumsum_dia / cambios.dpcapacity
cambios = agregar_caracteristicas_de_hora(cambios, BIN_MIN)

cambios["es_fin_semana"] = (cambios.bin.dt.dayofweek >= 5).astype(int)
FEATURES = ([f"neto_{k}_relativo" for k in (1, 2, 4, 8)] + [f"{c}_{k}" for c in ("salidas", "entradas") for k in (2, 4)] +
            ["cumsum_dia_relativo", "dpcapacity", "latitude", "longitude", "hora_sin", "hora_cos", "es_fin_semana"])

In [25]:
# Ajuste de para mejorar almacenamiento
columnas_salida = FEATURES + ["clase", "estacion", "dia", "hora_idx"]
cambios = cambios[columnas_salida]
for c in cambios.select_dtypes("integer").columns:
    cambios[c] = pd.to_numeric(cambios[c], downcast="integer")        # sin pérdida
for c in cambios.select_dtypes("float64").columns:       # 2. decimales a float32
    cambios[c] = cambios[c].astype("float32")

Con esto se procede al entrenamiento del modelo, en la libreta 0.04-fat-model-training-p2.ipynb.

In [26]:
# Se guardan los resultados limpios del dataframe de viajes:
cambios.to_parquet(os.path.join(DATA_DIR, "p2_datos_cambios_de_capacidad.parquet"))